# 04. Подготовка единой таблицы профилей для Entity Resolution

Приводим исходный датасет к прозрачному аналитическому виду:

1. Берём исходные строки событий/профилей.
2. Разворачиваем массивы `non_processing_features` и `fs_features`.
3. Разворачиваем JSON-поле `realtime_features`.
4. Собираем единую широкую таблицу уровня `profile_id`.
5. Сохраняем длинные token-level таблицы для аудита.
6. Формируем отчёт по колонкам.


In [ ]:
from pathlib import Path
import json
import re
from collections import Counter

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
DATA_PATH = Path("../data/")

INPUT_PATH =  DATA_PATH / "split_label_train_V3.snappy.parquet"
OUTPUT_DIR = DATA_PATH / "prepared_entity_resolution_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH, OUTPUT_DIR

## 1. Загружаем исходный датасет

In [ ]:
raw = pd.read_parquet(INPUT_PATH).reset_index(names='raw_row_id')
raw['created_at'] = pd.to_datetime(raw['created_at'], errors='coerce')

## 2. Вспомогательные функции

In [ ]:
EMPTY = {'', 'none', 'null', 'nan', '<na>'}


def is_empty(x) -> bool:
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass
    return isinstance(x, str) and x.strip().lower() in EMPTY


def mode_non_empty(s: pd.Series):
    vals = [str(x).strip() for x in s if not is_empty(x)]
    if not vals:
        return pd.NA
    c = Counter(vals)
    return sorted(c.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]

def normalize_text(s: pd.Series) -> pd.Series:
    return (
        s.astype('string')
         .str.strip()
         .str.lower()
         .replace({'': pd.NA, 'none': pd.NA, 'nan': pd.NA, 'null': pd.NA})
    )


def digits_only(x):
    if is_empty(x):
        return pd.NA
    d = re.sub(r'\D+', '', str(x))
    return d if d else pd.NA

## 3. Базовая таблица уровня `profile_id`

Техническая основа для сборки финальной `profile_flat_for_analysis`.

Здесь оставляем только поля, которые нужны дальше:
- `profile_id`;
- `entity_id` и `entity_size` как разметку для оценки качества;
- число событий и границы времени наблюдений;
- нормализованные identity-поля под обычными именами колонок.


In [ ]:
def build_profile_base(raw: pd.DataFrame) -> pd.DataFrame:
    g = raw.groupby('profile_id', dropna=False, sort=False)

    base = g.agg(
        event_count=('raw_row_id', 'size'),
        entity_id=('entity_id', mode_non_empty),
        first_event_at=('created_at', 'min'),
        last_event_at=('created_at', 'max'),
        first_name=('first_name', mode_non_empty),
        last_name=('last_name', mode_non_empty),
        email=('email', mode_non_empty),
        phone=('phone', mode_non_empty),
        birthday=('birthday', mode_non_empty),
        sex=('sex', mode_non_empty),
    ).reset_index()

    entity_size = (
        base.groupby('entity_id', dropna=False)['profile_id']
            .nunique()
            .rename('entity_size')
            .reset_index()
    )
    base = base.merge(entity_size, on='entity_id', how='left')

    base['first_name'] = normalize_text(base['first_name'])
    base['last_name'] = normalize_text(base['last_name'])
    base['sex'] = normalize_text(base['sex']).replace({'unknown': pd.NA})
    base['email'] = normalize_text(base['email'])
    base['email_domain'] = base['email'].str.extract(r'@(.+)$', expand=False)
    base['phone'] = base['phone'].map(digits_only).astype('string')

    birthday_dt = pd.to_datetime(base['birthday'], errors='coerce')
    base['birthday'] = birthday_dt.dt.strftime('%Y-%m-%d').astype('string')

    keep_cols = [
        'profile_id', 'event_count', 'entity_id', 'entity_size',
        'first_event_at', 'last_event_at',
        'first_name', 'last_name', 'sex',
        'email', 'email_domain', 'phone', 'birthday',
    ]
    return base[keep_cols]

profile_base = build_profile_base(raw)
display(profile_base.head(1))


## 4. Разворачиваем `non_processing_features` и `fs_features` в длинный формат

Длинный формат нужен, чтобы получить пары `feature_name / feature_value` и дальше строить из них wide-таблицу, словарь признаков и blocking-анализ.


In [ ]:
def explode_array(raw: pd.DataFrame, col: str, source: str) -> pd.DataFrame:
    x = raw[['raw_row_id', 'profile_id', 'entity_id', 'created_at', col]].explode(col, ignore_index=True)
    x = x.dropna(subset=[col]).rename(columns={col: 'raw_feature'})
    x['raw_feature'] = x['raw_feature'].astype('string').str.strip()
    x = x[x['raw_feature'].notna() & x['raw_feature'].ne('')]

    parts = x['raw_feature'].str.split(':', n=1, expand=True)
    x['source'] = source
    x['feature_name'] = parts[0].astype('string').str.strip()
    x['feature_value'] = parts[1].astype('string').str.strip() if parts.shape[1] > 1 else pd.NA
    x.loc[x['feature_value'].eq(''), 'feature_value'] = pd.NA

    return x[[
        'raw_row_id', 'profile_id', 'entity_id', 'created_at',
        'source', 'feature_name', 'feature_value'
    ]]


np_long = explode_array(raw, 'non_processing_features', 'non_processing_features')
fs_long = explode_array(raw, 'fs_features', 'fs_features')

print('non_processing long:', np_long.shape)
print('fs long:', fs_long.shape)
display(np_long.head(3))
display(fs_long.head(3))


## 5. Разворачиваем `realtime_features`

`realtime_features` хранится как JSON-строка. Его тоже переводим в длинную таблицу `feature_name / feature_value`.

In [ ]:
def parse_realtime_payload(payload) -> dict:
    if is_empty(payload):
        return {}
    try:
        d = json.loads(payload) if isinstance(payload, str) else payload
    except Exception:
        return {}
    return d if isinstance(d, dict) else {}


def explode_realtime(raw: pd.DataFrame) -> pd.DataFrame:
    records = []
    cols = ['raw_row_id', 'profile_id', 'entity_id', 'created_at', 'realtime_features']
    for raw_row_id, profile_id, entity_id, created_at, payload in raw[cols].itertuples(index=False, name=None):
        d = parse_realtime_payload(payload)
        for k, v in d.items():
            if v is not None:
                records.append((
                    raw_row_id, profile_id, entity_id, created_at,
                    'realtime_features', str(k), str(v)
                ))
    return pd.DataFrame(
        records,
        columns=['raw_row_id', 'profile_id', 'entity_id', 'created_at', 'source', 'feature_name', 'feature_value']
    )


rt_long = explode_realtime(raw)
print('realtime long:', rt_long.shape)
display(rt_long.head())

## 6. Собираем минимальные широкие блоки признаков

Широкая таблица используется как удобный плоский слой для последующего анализа и проверки blocking.

Оставляем только raw-признаки, из которых дальше реально строятся blocking-правила, derived-ключи или similarity-признаки:

- identity уже лежит в `profile_base`: `email`, `phone`, ФИО, дата рождения, пол. Из `email` дальше строятся `email_domain`, `email_initial2`, `email_hash_bucket_1024`; `phone` используется для `identity_rescue`;
- `np`: `geoname_id`, `subdivision_1_iso_code`, `device`, `osfamily`, `browser`;
- `rt`: `geoname`, `local_hour`, `day`; `geoid` оставляем как вспомогательный geo-признак для similarity/сравнения, но он не входит в текущий recommended blocking;
- `fs`: основные behavior-признаки, postman-признаки и site-id/событийные признаки, которые участвуют в актуальных правилах.

Важно: derived-признаки `registration_60m_bucket`, `rt_daypart`, `rt_weekpart` здесь не хранятся отдельными колонками. Они строятся позже из `created_at`, `rt__local_hour` и `rt__day`.


In [ ]:
WIDE_FEATURES = {
    'np': [
        'browser',
        'device',
        'geoname_id',
        'osfamily',
        'subdivision_1_iso_code',
    ],
    'rt': [
        'day',
        'geoid',
        'geoname',
        'local_hour',
    ],
    'fs': [
        'source_site_365', 'source_site_30',
        'visited_30', 'visited_365',
        'has_account',
        'has_click_365', 'has_click_30',
        'has_accept_365', 'has_accept_30',
        'has_order_365', 'has_order_30',
        'postman_action_90', 'postman_action_30',
        'postman_campaign_90', 'postman_campaign_30',
        'postman_response_90', 'postman_response_30',
    ],
}


def wide_selected_values(long_df: pd.DataFrame, prefix: str, features: list[str]) -> pd.DataFrame:
    if long_df.empty:
        return pd.DataFrame(columns=['profile_id'])

    val = long_df[
        long_df['feature_value'].notna()
        & long_df['feature_name'].isin(features)
    ][['profile_id', 'feature_name', 'feature_value']].copy()

    if val.empty:
        return pd.DataFrame(columns=['profile_id'])

    val['feature_value'] = val['feature_value'].astype(str)
    counts = (
        val.groupby(['profile_id', 'feature_name', 'feature_value'], sort=False)
           .size()
           .reset_index(name='cnt')
    )

    mode = (
        counts.sort_values(
            ['profile_id', 'feature_name', 'cnt', 'feature_value'],
            ascending=[True, True, False, True],
            kind='mergesort',
        )
        .drop_duplicates(['profile_id', 'feature_name'])
        .pivot(index='profile_id', columns='feature_name', values='feature_value')
        .reset_index()
    )

    mode.columns = [
        'profile_id' if c == 'profile_id' else f'{prefix}__{c}'
        for c in mode.columns
    ]
    return mode


def coerce_realtime_types(rt_wide: pd.DataFrame) -> pd.DataFrame:
    for c in rt_wide.columns:
        if c == 'profile_id':
            continue

        num = pd.to_numeric(rt_wide[c], errors='coerce')
        non_missing = rt_wide[c].notna().sum()
        if non_missing and num.notna().sum() / non_missing >= 0.95:
            if np.allclose(num.dropna(), np.round(num.dropna())):
                rt_wide[c] = num.astype('Int64')
            else:
                rt_wide[c] = num
        else:
            rt_wide[c] = rt_wide[c].astype('string')
    return rt_wide


np_wide = wide_selected_values(np_long, 'np', WIDE_FEATURES['np'])
fs_wide = wide_selected_values(fs_long, 'fs', WIDE_FEATURES['fs'])
rt_wide = coerce_realtime_types(wide_selected_values(rt_long, 'rt', WIDE_FEATURES['rt']))

print('np_wide:', np_wide.shape)
print('fs_wide:', fs_wide.shape)
print('rt_wide:', rt_wide.shape)


In [ ]:
fs_wide.head(3)

## 7. Финальная таблица `profile_flat_for_analysis`

Строим финальную таблицу на основании которой можно проводить анализ и строить правила.
- минимальный identity-блок в нормализованном виде;
- развернутые `non_processing_features`;
- развернутые `fs_features`;
- развернутые `realtime_features`;
- служебные поля для контроля качества.
Замечу `entity_id` оставляем только как label/ground truth для анализа.

In [ ]:
flat = (
    profile_base
    .merge(np_wide, on='profile_id', how='left')
    .merge(rt_wide, on='profile_id', how='left')
    .merge(fs_wide, on='profile_id', how='left')
)

for c in [c for c in flat.columns if c.endswith(('__cnt', '__nunique'))]:
    flat[c] = flat[c].fillna(0).astype('int32')

for c in [c for c in flat.columns if c.endswith('__has')]:
    flat[c] = flat[c].astype('boolean').fillna(False).astype(bool)

print(flat.shape)
display(flat.head(2))

## 8. Сохраняем результаты

Сохраняем только основной артефакт этого ноутбука — плоскую таблицу уровня `profile_id`.
- `profile_flat_for_analysis.parquet` — основная широкая таблица уровня `profile_id`.


In [ ]:
flat.to_parquet(OUTPUT_DIR / 'profile_flat_for_analysis.parquet', index=False)


summary = {
    'input': str(INPUT_PATH),
    'raw_rows': int(len(raw)),
    'profile_rows': int(len(flat)),
    'entity_count': int(flat['entity_id'].nunique()),
    'flat_columns': int(flat.shape[1]),
    'output_dir': str(OUTPUT_DIR),
}

summary